In [0]:
# Imports
from effodata import ACDS, golden_rules, Joiner, Sifter, Equality, join_on
from kpi_metrics import KPI, AliasMetric, CustomMetric, AliasGroupby, get_metrics, available_metrics, Rollup, Cube
import pyspark.sql.functions as f
from pyspark.sql.types import *
import re
import os
import sys
import time
import upc_input
import datetime as dt
import seg
from seg import profile
from poirot import SparkManager

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
kpi = KPI(use_sample_mart = False, apply_privacy_filters = True)
acds = ACDS(use_sample_mart = False, apply_privacy_filters = True)

#### Campaign Availability Tab of KPF Dashboard.

- Contains a list of all KPF Projects, Channel, Business Line, Measurement Kickoff date, as well as Measurement Status. Used to check for status of completed measurement runs, measurements in-progress, and failures.

<br>

<img src="./Campaign_Availability_Tab_Image_Sample.png" alt="Campaign_Availability_Tab_Image_Sample.png" title="Campaign_Availability_Tab_Image_Sample.png" width="1000"/>

In [0]:
# Code to generate campaign availability tab.

# Reminder to use a Non-UC enabled cluster when pulling KPM metadata.
media_history_revamped = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')

# hard Coding KPM Campaig 93220 = MMCI table has this campaign mislabeled, should be in KPF line.
mmci = mmci.withColumn(
  "MANUFACTURER", f.when(f.col("KPM_PROJECT_ID") == 93220, "Kroger Personal Finance").otherwise(f.col("MANUFACTURER"))
)

# Filter to necessary KPF campaigns and relevant channels
mmci = mmci.withColumn(
    'Fiscal_Quarter',
    f.concat(f.lit('Q'), f.substring(f.col('START_FIS_QUARTER_NAME'), 9, 1))
).filter(
  (f.col('KROGER_START_WEEK') >= '20220101') &
  (f.lower(f.col('MANUFACTURER')).isin(['kroger personal finance','kroger wallet'])) & 
  (f.col('channel').isin(['Display Ad', 'Targeted Digital Coupon', 'Single Subject Email', 'Pandora', 'Pinterest', 'Push Notifications', 'Pre-Roll Video', 'Email Module'])) & 
  (f.col("SIGNED_OFF") == "Y") 
)

campaign_availability = (
    mmci.select(
        'Fiscal_Start_Year','Fiscal_Quarter','kpm_duplicated_id','kpm_project_id',
        'project_name','Channel','incr_trigger_date','camp_start_date',
        'camp_end_date','tot_cost'
    )
    .withColumn('kpm_duplicated_id', f.col('kpm_duplicated_id').cast('string'))
    .join(
        media_history_revamped
            .select('campaign_id', 'job_id')
            .distinct()
            .withColumnRenamed('campaign_id', 'kpm_duplicated_id'),
        ['kpm_duplicated_id'],
        'left'
    )
)

campaign_availability.display()

In [0]:
campaign_availability = campaign_availability.withColumn(
    "business_line",
    f.when(
        f.upper(f.col("project_name")).contains("OPEN LOOP") | 
        f.upper(f.col("project_name")).contains(" OL "), "Open Loop"
    ).when(
        f.upper(f.col("project_name")).contains("LOCAL") |
        f.upper(f.col("project_name")).contains("TDC KPF") |
        f.upper(f.col("project_name")).contains("TDC SFID") |
        f.upper(f.col("project_name")).contains("TDC SFPRJ") |
        f.upper(f.col("project_name")).contains("MCP") |
        f.upper(f.col("project_name")).contains("BULK") |
        f.upper(f.col("project_name")).contains("GIFT"), "Gift"
    ).when(
        f.upper(f.col("project_name")).contains("LOTT"), "Lottery"
    ).when(
        f.upper(f.col("project_name")).contains("MONEY SERVICES") |
        f.upper(f.col("project_name")).contains(" MS "), "Money Services"
    ).when(
        f.upper(f.col("project_name")).contains(" PAY "), "Kroger Pay"
    ).when(
        f.upper(f.col("project_name")).contains("KROGER WALLET"), "Kroger Wallet"
    ).when(
        f.upper(f.col("project_name")).contains(" ACH "), "Debit"
    ).when(
        f.upper(f.col("project_name")).contains(" CICO "), "Account Funding"
    ).when(
        f.upper(f.col("project_name")).contains(" SBE "), "SBE"
    ).when(
        f.upper(f.col("project_name")).contains("CREDIT"), "Credit"
    ).when(
        f.upper(f.col("project_name")).contains(" REM "), "Gift"
    ).otherwise("")
)

campaign_availability = campaign_availability.filter(f.col("camp_start_date") >= "2023-01-01")

campaign_availability.display()

In [0]:
campaign_availability.coalesce(1).write.mode("overwrite").option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/campaign_availability_tab.csv')

campaign_availability_read = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/campaign_availability_tab.csv')
campaign_availability_read.display()